In [15]:
import numpy as np
import pandas as pd
import re

pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', None) # so i can activate scrollable elements

In [6]:
data = pd.read_csv("laptops.csv", encoding="latin1")
data.head()

,Manufacturer,Model Name,Category,Screen Size,Screen,CPU,RAM,Storage,GPU,Operating System,Operating System Version,Weight,Price (Euros)
0,Apple,MacBook Pro,Ultrabook,"13.3""",IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,NaN,1.37kg,"1339,69"
1,Apple,Macbook Air,Ultrabook,"13.3""",1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,NaN,1.34kg,"898,94"
2,HP,250 G6,Notebook,"15.6""",Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,NaN,1.86kg,"575,00"
3,Apple,MacBook Pro,Ultrabook,"15.4""",IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,NaN,1.83kg,"2537,45"
4,Apple,MacBook Pro,Ultrabook,"13.3""",IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,NaN,1.37kg,"1803,60"


In [7]:
data.shape

(1303, 13)

In [8]:
USAGE_PRIORITIES = {
    "Bureautique": ["Weight", "Price (Euros)"],
    "Gaming": ["GPU", "CPU", "RAM"],
    "Création": ["Screen Size", "CPU", "RAM"]
}


GPU_HIERARCHY = {
    "Faible": ["Intel Iris Xe"],
    "Moyen": ["NVIDIA GTX", "AMD Radeon RX 6500M"],
    "Élevé": ["NVIDIA RTX 40", "AMD Radeon RX 7700M"]
}

In [9]:
def recommend_constraint(df, brand=None, min_ram=None, max_price=None):
  
    result = df.copy()

    if brand is not None:
        result = result[result['Manufacturer'] == brand]
    if min_ram is not None:
        result = result[result['RAM'] >= min_ram]
    if max_price is not None:
        result = result[result['Price (Euros)'] <= max_price]

    if result.empty:
        result = df.copy()
       
        if min_ram is not None:
            result = result[result['RAM'] >= min_ram]
        if max_price is not None:
            result = result[result['Price (Euros)'] <= max_price * 1.15] 
    return result

In [10]:
df_laptops = data.copy()
df_laptops['RAM'] = df_laptops['RAM'].astype(str).str.extract(r'(\d+)').astype(int)
df_laptops['Price (Euros)'] = df_laptops['Price (Euros)'].astype(str).str.replace(',', '.').str.extract(r'([\d\.]+)').astype(float)



In [11]:
recommandations = recommend_constraint(df_laptops, brand="Dell", min_ram=16, max_price=800)
print(recommandations)

    Manufacturer     Model Name  Category Screen Size  \
226         Dell  Inspiron 5567  Notebook       15.6"   
235         Dell  Inspiron 5567  Notebook       15.6"   
611         Dell  Inspiron 7559    Gaming       15.6"   

                              Screen                          CPU  RAM  \
226  Full HD / Touchscreen 1920x1080   Intel Core i7 7500U 2.7GHz   16   
235                Full HD 1920x1080   Intel Core i7 7500U 2.7GHz   16   
611                Full HD 1920x1080  Intel Core i7 6700HQ 2.6GHz   16   

       Storage                             GPU Operating System  \
226    1TB HDD              AMD Radeon R7 M445          Windows   
235  256GB SSD              AMD Radeon R7 M445            Linux   
611    1TB HDD  Nvidia GeForce GTX 960<U+039C>          Windows   

    Operating System Version  Weight  Price (Euros)  
226                       10   2.3kg         859.01  
235                      NaN  2.36kg         899.00  
611                       10  2.59kg       

In [12]:
df_laptops = data.copy().reset_index(drop=True)
df_laptops["laptop_id"] = df_laptops.index

In [13]:
def extract_number(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"\d+(?:[.,]\d+)?", str(value).replace(",", "."))
    return float(match.group(0)) if match else np.nan


In [16]:
df_laptops["RAM_num"] = df_laptops["RAM"].apply(extract_number)
df_laptops["Price_num"] = df_laptops["Price (Euros)"].apply(extract_number)
df_laptops["Weight_num"] = df_laptops["Weight"].apply(extract_number)
df_laptops["ScreenSize_num"] = df_laptops["Screen Size"].apply(extract_number)

In [ ]:
def normalize_01(series):
    s = pd.to_numeric(series, errors="coerce")
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi):
        return pd.Series(np.nan, index=series.index)
    if hi == lo:
        return pd.Series(np.where(s.notna(), 1.0, np.nan), index=series.index)
    return (s - lo) / (hi - lo)

for col in ["RAM_num", "Price_num", "Weight_num", "ScreenSize_num"]:
    df_laptops[col + "_norm"] = normalize_01(df_laptops[col])

In [18]:
def gpu_level(value):
    s = str(value).lower()
    if any(x in s for x in ["rtx", "radeon rx 7", "geforce rtx"]):
        return 3
    if any(x in s for x in ["gtx", "radeon rx", "radeon pro", "quadro"]):
        return 2
    if any(x in s for x in ["iris", "uhd", "hd graphics", "vega", "integrated"]):
        return 1
    return np.nan

In [19]:
def categorical_similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return float(str(a).strip().lower() == str(b).strip().lower())

def gpu_similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    la, lb = gpu_level(a), gpu_level(b)
    if pd.notna(la) and pd.notna(lb):
        return 1 - abs(la - lb) / 2
    return categorical_similarity(a, b)

In [20]:
def cpu_similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    sa, sb = str(a).lower(), str(b).lower()
    # La marque/famille du processeur fournit une proximité partielle, même si les modèles diffèrent.
    if sa == sb:
        return 1.0
    if ("intel" in sa and "intel" in sb) or ("amd" in sa and "amd" in sb):
        return 0.5
    return 0.0

In [21]:
USAGE_WEIGHTS = {
    "Bureautique": {
        "Weight_num_norm": 0.25, "Price_num_norm": 0.25,
        "RAM_num_norm": 0.10, "Battery": 0.20,
        "Screen Size": 0.10, "Operating System": 0.10
    },
    "Gaming": {
        "GPU": 0.35, "CPU": 0.25, "RAM_num_norm": 0.20,
        "Screen Size": 0.10, "Price_num_norm": 0.05, "Weight_num_norm": 0.05
    },
    "Création": {
        "Screen Size": 0.25, "CPU": 0.25, "RAM_num_norm": 0.20,
        "GPU": 0.15, "Price_num_norm": 0.10, "Weight_num_norm": 0.05
    }
}

In [22]:
def _numeric_similarity(a, b, lower_is_better=False):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    # Les valeurs sont déjà normalisées dans [0,1].
    sim = 1 - abs(float(a) - float(b))
    return float(np.clip(sim, 0, 1))

def _attribute_similarity(row_a, row_b, attr):
    if attr.endswith("_norm"):
        lower_is_better = attr in ("Price_num_norm", "Weight_num_norm")
        return _numeric_similarity(row_a.get(attr), row_b.get(attr), lower_is_better)
    if attr == "GPU":
        return gpu_similarity(row_a.get(attr), row_b.get(attr))
    if attr == "CPU":
        return cpu_similarity(row_a.get(attr), row_b.get(attr))
    return categorical_similarity(row_a.get(attr), row_b.get(attr))

In [23]:
def recommend_similar(laptop_id, usage_scenario, weights=None, top_n=10, include_scores=True):
    if usage_scenario not in USAGE_WEIGHTS and weights is None:
        raise ValueError(f"Scénario inconnu : {usage_scenario}. Choisir parmi {list(USAGE_WEIGHTS)}")
    if "laptop_id" not in df_laptops.columns:
        raise ValueError("La colonne laptop_id est absente.")
    target_rows = df_laptops[df_laptops["laptop_id"] == laptop_id]
    if target_rows.empty:
        raise ValueError(f"Aucun laptop trouvé pour laptop_id={laptop_id}")
    target = target_rows.iloc[0]
    active_weights = weights if weights is not None else USAGE_WEIGHTS[usage_scenario]
    active_weights = {k: float(v) for k, v in active_weights.items() if k in df_laptops.columns and v > 0}
    if not active_weights:
        raise ValueError("Aucun poids valide ne correspond aux colonnes du catalogue.")
    weight_sum = sum(active_weights.values())
    if weight_sum <= 0:
        raise ValueError("La somme des poids doit être positive.")


In [31]:

def recommend_similar(laptop_id, usage_scenario="Bureautique",
                      weights=None, top_n=10, include_scores=True):

    # Vérifier que l'ordinateur existe
    target_rows = df_laptops[df_laptops["laptop_id"] == laptop_id]

    if target_rows.empty:
        raise ValueError(f"Aucun ordinateur trouvé avec l'ID {laptop_id}")

    target = target_rows.iloc[0]

    # Définir les poids selon le scénario d'utilisation
    if weights is not None:
        active_weights = weights
    else:
        active_weights = USAGE_WEIGHTS[usage_scenario]

    rows = []

    # Comparer l'ordinateur cible aux autres ordinateurs
    for _, candidate in df_laptops.iterrows():

        if candidate["laptop_id"] == laptop_id:
            continue  # Ne pas recommander le produit lui-même

        weighted_sum = 0.0
        available_weight = 0.0
        details = {}

        for attr, weight in active_weights.items():
            sim = _attribute_similarity(target, candidate, attr)
            details[f"sim_{attr}"] = sim

            if pd.notna(sim):
                weighted_sum += weight * sim
                available_weight += weight

        score = (
            weighted_sum / available_weight
            if available_weight > 0
            else np.nan
        )

        rows.append({
            "laptop_id": int(candidate["laptop_id"]),
            "Manufacturer": candidate.get("Manufacturer"),
            "Model Name": candidate.get("Model Name"),
            "Category": candidate.get("Category"),
            "RAM": candidate.get("RAM"),
            "GPU": candidate.get("GPU"),
            "CPU": candidate.get("CPU"),
            "Price (Euros)": candidate.get("Price (Euros)"),
            "similarity_score": score,
            **details
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    result = result.dropna(subset=["similarity_score"])
    result = result.sort_values(
        "similarity_score",
        ascending=False
    )

    if include_scores:
        return result.head(top_n)

    return result.head(top_n).drop(
        columns=[
            c for c in result.columns
            if c.startswith("sim_")
        ]
    )


In [32]:
zephyrus = df_laptops[df_laptops["Model Name"].astype(str).str.contains("Zephyrus", case=False, na=False)]
display(zephyrus[["laptop_id", "Manufacturer", "Model Name", "Category", "RAM", "CPU", "GPU", "Price (Euros)"]])

if not zephyrus.empty:
    reference_id = int(zephyrus.iloc[0]["laptop_id"])
    print("ID utilisé pour le modèle Zephyrus :", reference_id)
else:
    reference_id = 4
    print("Aucun modèle Zephyrus trouvé. Vérifiez la ligne d'ID 4 avant d'interpréter les résultats.")
    display(df_laptops.loc[df_laptops["laptop_id"] == reference_id,
                           ["laptop_id", "Manufacturer", "Model Name", "Category", "RAM", "CPU", "GPU", "Price (Euros)"]])

# Requête 2 : rechercher des machines similaires pour un usage Création.
if reference_id in df_laptops["laptop_id"].values:
    recommandations_creation = recommend_similar(reference_id, "Création", top_n=3)
    display(recommandations_creation)

,laptop_id,Manufacturer,Model Name,Category,RAM,CPU,GPU,Price (Euros)
517,517,Asus,ROG Zephyrus,Gaming,24GB,Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX1080,"2968,00"


ID utilisé pour le modèle Zephyrus : 517


,laptop_id,Manufacturer,Model Name,Category,RAM,GPU,CPU,Price (Euros),similarity_score,sim_Screen Size,sim_CPU,sim_RAM_num_norm,sim_GPU,sim_Price_num_norm,sim_Weight_num_norm
329,329,Dell,XPS 15,Notebook,32GB,Nvidia GeForce GTX 1050,Intel Core i7 7700HQ 2.8GHz,"2639,00",0.966396,1.0,1.0,0.870968,1.0,0.944473,0.955112
1208,1209,Asus,Rog Strix,Gaming,16GB,Nvidia GeForce GTX 1070,Intel Core i7 7700HQ 2.8GHz,"2449,00",0.964935,1.0,1.0,0.870968,1.0,0.912405,0.990025
638,639,Dell,XPS 15,Notebook,16GB,Nvidia GeForce GTX 1050,Intel Core i7 7700HQ 2.8GHz,"2399,00",0.962346,1.0,1.0,0.870968,1.0,0.903966,0.955112


In [33]:
def recommend_constraint(df, brand=None, min_ram=None, max_price=None, relax=True, price_strict=True):
    """Filtrage conjonctif. brand accepte une chaîne ou une liste de marques.
    Si aucun résultat, relaxe progressivement les contraintes et indique ce qui a été relâché.
    """
    work = df.copy()
    if "RAM" not in work.columns or "Price (Euros)" not in work.columns:
        raise ValueError("Le DataFrame doit contenir RAM et Price (Euros).")
    # Normalisation locale pour éviter de dépendre de la cellule précédente.
    ram = work["RAM"].apply(extract_number)
    price = work["Price (Euros)"].apply(extract_number)
    brands = [brand] if isinstance(brand, str) else list(brand) if brand is not None else None

    def apply_filters(use_brand=True, use_ram=True, use_price=True, price_limit=None):
        mask = pd.Series(True, index=work.index)
        if use_brand and brands:
            mask &= work["Manufacturer"].astype(str).str.casefold().isin([b.casefold() for b in brands])
        if use_ram and min_ram is not None:
            mask &= ram >= min_ram
        if use_price and max_price is not None:
            limit = max_price if price_limit is None else price_limit
            mask &= price < limit if price_strict else price <= limit
        return work.loc[mask].copy()

    result = apply_filters()
    result.attrs["relaxation"] = "Aucune"
    if not result.empty or not relax:
        return result

    # Relaxation 1 : marques, puis relaxation 2 : prix (+15 %), puis RAM.
    result = apply_filters(use_brand=False)
    if not result.empty:
        result.attrs["relaxation"] = "Marque(s) relâchée(s) ; RAM et prix conservés"
        return result

    result = apply_filters(use_brand=False, use_price=True, price_limit=max_price * 1.15 if max_price is not None else None)
    if not result.empty:
        result.attrs["relaxation"] = "Marque(s) relâchée(s), plafond de prix augmenté de 15 % ; RAM conservée"
        return result

    result = apply_filters(use_brand=False, use_ram=False, use_price=True,
                           price_limit=max_price * 1.15 if max_price is not None else None)
    result.attrs["relaxation"] = "Marque(s) et RAM relâchées ; plafond de prix augmenté de 15 %"
    return result

# Requête 1 : Dell OU Lenovo, RAM >= 16 Go, prix < 1700 €.
resultat_contraintes = recommend_constraint(
    df_laptops, brand=["Dell", "Lenovo"], min_ram=16, max_price=1700, relax=True
)
print("Nombre de résultats :", len(resultat_contraintes))
print("Relaxation appliquée :", resultat_contraintes.attrs.get("relaxation", "Non renseignée"))
display(resultat_contraintes[["Manufacturer", "Model Name", "RAM", "Price (Euros)", "CPU", "GPU"]].head(20))

Nombre de résultats : 41
Relaxation appliquée : Aucune


,Manufacturer,Model Name,RAM,Price (Euros),CPU,GPU
41,Dell,Inspiron 7577,16GB,"1499,00",Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1060
60,Dell,Inspiron 5770,16GB,"1299,00",Intel Core i7 8550U 1.8GHz,AMD Radeon 530
85,Dell,Inspiron 7577,16GB,"1349,00",Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1050 Ti
128,Dell,Inspiron 5770,16GB,"1396,00",Intel Core i7 8550U 1.8GHz,AMD Radeon 530
150,Dell,Inspiron 5577,16GB,"1249,26",Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1050
163,Lenovo,Legion Y520-15IKBN,16GB,"1079,00",Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1050 Ti
180,Dell,Inspiron 5570,16GB,"1049,00",Intel Core i7 8550U 1.8GHz,AMD Radeon 530
185,Dell,Inspiron 7773,16GB,"1549,00",Intel Core i7 8550U 1.8GHz,Nvidia GeForce 150MX
205,Lenovo,Legion Y520-15IKBN,16GB,"1398,00",Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1060
219,Dell,Inspiron 5379,16GB,"1279,00",Intel Core i7 8550U 1.8GHz,Intel UHD Graphics 620


In [ ]:
if not resultat_contraintes.empty and resultat_contraintes.attrs.get("relaxation") == "Aucune":
    ram_ok = resultat_contraintes["RAM"].apply(extract_number).ge(16).all()
    prix_ok = resultat_contraintes["Price (Euros)"].apply(extract_number).lt(1700).all()
    marque_ok = resultat_contraintes["Manufacturer"].astype(str).str.casefold().isin(["dell", "lenovo"]).all()
    print(f"Respect RAM >= 16 Go : {ram_ok}")
    print(f"Respect prix < 1700 € : {prix_ok}")
    print(f"Respect marque Dell/Lenovo : {marque_ok}")
elif resultat_contraintes.empty:
    print("Aucun résultat, même après les étapes de relaxation.")
else:
    print("Les résultats sont relaxés : vérifier la description affichée avant de conclure qu'ils satisfont la requête.")

if "recommandations_creation" in globals() and not recommandations_creation.empty:
    cols = ["laptop_id", "Manufacturer", "Model Name", "similarity_score"]
    print("Top 3 — scénario Création")
    display(recommandations_creation[cols].reset_index(drop=True))
    premier = recommandations_creation.iloc[0]
    print(f"Le premier résultat a un score global de {premier['similarity_score']:.3f}.")
    print("Le score est la moyenne pondérée des similarités disponibles pour l'écran, le CPU, la RAM, le GPU, le prix et le poids.")
else:
    print("Exécutez d'abord la cellule de recherche du ROG Zephyrus pour calculer le Top 3.")

Respect RAM >= 16 Go : True
Respect prix < 1700 € : True
Respect marque Dell/Lenovo : True
Top 3 — scénario Création


,laptop_id,Manufacturer,Model Name,similarity_score
0,329,Dell,XPS 15,0.966396
1,1209,Asus,Rog Strix,0.964935
2,639,Dell,XPS 15,0.962346


Le premier résultat a un score global de 0.966.
Le score est la moyenne pondérée des similarités disponibles pour l'écran, le CPU, la RAM, le GPU, le prix et le poids.


# Analyse des résultats

## 1. Système à contraintes — Requête 1

Le système recherche les laptops respectant les critères spécifiés. Si aucun résultat ne satisfait toutes les contraintes, une relaxation permet d'assouplir certains critères pour proposer des alternatives.

**Résultat :** [Fonctionnement sans relaxation / Relaxation nécessaire.]

## 2. Système sémantique — Requête 2

Les trois premiers laptops recommandés sont :

1. **[Modèle 1]** — Score : [score]
2. **[Modèle 2]** — Score : [score]
3. **[Modèle 3]** — Score : [score]

Le premier laptop possède le score global le plus élevé, calculé à partir des similarités individuelles et des poids attribués aux caractéristiques selon le scénario d'utilisation.

## 3. Limites et améliorations

Le système à contraintes dépend des critères explicites, tandis que le système sémantique dépend des caractéristiques disponibles et des poids choisis. La base pourrait être enrichie avec les avis des utilisateurs, les performances réelles, l'autonomie, les prix actualisés et les préférences des acheteurs.

## Conclusion

Les deux approches sont complémentaires : le système à contraintes filtre les produits selon les exigences, tandis que le système sémantique classe les produits selon leur similarité.